# Customer Churn Prediction
XGBoost + SHAP on the Telco Customer Churn dataset.

**Data:** Kaggle *Telco Customer Churn* (`WA_Fn-UseC_-Telco-Customer-Churn.csv`). On Kaggle, add the dataset to the notebook and change `PATH` in Step 1 to `/kaggle/input/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv`. On Colab or locally, upload the CSV next to the notebook.

Run the cells top to bottom.

In [ ]:
!pip install -q pandas numpy scikit-learn xgboost shap matplotlib

## Step 1: load

In [ ]:
import json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import shap
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             precision_recall_curve, recall_score,
                             precision_score, confusion_matrix)

SEED = 42
np.random.seed(SEED)
PATH = "WA_Fn-UseC_-Telco-Customer-Churn.csv"
results = {}

df = pd.read_csv(PATH)
print(df.shape)
print(df["Churn"].value_counts(normalize=True).round(3))   # ~26.5% churn: imbalanced

## Step 2: clean

In [ ]:
# TotalCharges has blank strings for brand-new customers (tenure = 0)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("blank TotalCharges:", df["TotalCharges"].isna().sum())
df["TotalCharges"] = df["TotalCharges"].fillna(0)          # they haven't been billed yet

df["Churn"] = (df["Churn"] == "Yes").astype(int)
df = df.drop(columns=["customerID"])                       # identifier, not a signal
df["SeniorCitizen"] = df["SeniorCitizen"].astype(int)

## Step 3: quick exploration (what drives churn?)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
df.groupby("Contract")["Churn"].mean().plot.bar(ax=axes[0], title="Churn rate by contract")
df.groupby("InternetService")["Churn"].mean().plot.bar(ax=axes[1], title="Churn rate by internet service")
df.groupby(pd.cut(df["tenure"], [0, 6, 12, 24, 48, 72], include_lowest=True))["Churn"].mean() \
    .plot.bar(ax=axes[2], title="Churn rate by tenure (months)")
plt.tight_layout(); plt.savefig("eda_churn_rates.png", dpi=120)

## Step 4: encode + split BEFORE any tuning (train / validation / test)

In [ ]:
X = pd.get_dummies(df.drop(columns=["Churn"]), drop_first=True).astype(float)
y = df["Churn"]

X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.25, stratify=y_tmp, random_state=SEED)   # 60/20/20
print(len(X_train), len(X_val), len(X_test))

## Step 5: baseline (logistic regression), then XGBoost

In [ ]:
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced"))
lr.fit(X_train, y_train)
lr_val = lr.predict_proba(X_val)[:, 1]

neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
xgb = XGBClassifier(
    n_estimators=800, learning_rate=0.03, max_depth=3,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=neg / pos,                     # handle the class imbalance
    eval_metric="aucpr", early_stopping_rounds=50,
    random_state=SEED)
xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
xgb_val = xgb.predict_proba(X_val)[:, 1]

val_scores = {
    "logreg_roc_auc": roc_auc_score(y_val, lr_val),
    "logreg_pr_auc": average_precision_score(y_val, lr_val),
    "xgb_roc_auc": roc_auc_score(y_val, xgb_val),
    "xgb_pr_auc": average_precision_score(y_val, xgb_val),
    "xgb_best_iteration": int(xgb.best_iteration),
}
print(json.dumps(val_scores, indent=2))
results["validation"] = val_scores

# Sturdier check: 5-fold CV ROC-AUC on the training portion
cv_auc = cross_val_score(
    XGBClassifier(n_estimators=int(xgb.best_iteration) + 1, learning_rate=0.03, max_depth=3,
                  subsample=0.8, colsample_bytree=0.8, scale_pos_weight=neg / pos,
                  random_state=SEED),
    X_tmp, y_tmp, cv=StratifiedKFold(5, shuffle=True, random_state=SEED), scoring="roc_auc")
results["xgb_cv_roc_auc_mean"] = float(cv_auc.mean())
print("CV ROC-AUC:", cv_auc.round(3), "mean", round(cv_auc.mean(), 3))

## Step 6: pick the decision threshold on VALIDATION (business-driven)

In [ ]:
# Assumption (edit to fit your story): contacting a customer with a retention offer
# costs 1 unit; losing a customer costs 5 units; an offer saves 40% of those who would leave.
COST_CONTACT, COST_LOSS, SAVE_RATE = 1.0, 5.0, 0.40

def net_value(y_true, probs, thr):
    flagged = probs >= thr
    saved = ((flagged) & (y_true == 1)).sum() * SAVE_RATE * COST_LOSS
    spent = flagged.sum() * COST_CONTACT
    return saved - spent

thresholds = np.linspace(0.05, 0.95, 91)
vals = [net_value(y_val.values, xgb_val, t) for t in thresholds]
threshold = float(thresholds[int(np.argmax(vals))])
print("chosen threshold:", round(threshold, 2))
results["threshold"] = threshold
results["cost_assumptions"] = {"contact": COST_CONTACT, "loss": COST_LOSS, "save_rate": SAVE_RATE}

## Step 7: evaluate ONCE on the test set

In [ ]:
xgb_test = xgb.predict_proba(X_test)[:, 1]
lr_test = lr.predict_proba(X_test)[:, 1]
pred = (xgb_test >= threshold).astype(int)

results["test"] = {
    "logreg_roc_auc": roc_auc_score(y_test, lr_test),
    "xgb_roc_auc": roc_auc_score(y_test, xgb_test),
    "xgb_pr_auc": average_precision_score(y_test, xgb_test),
    "recall": recall_score(y_test, pred),
    "precision": precision_score(y_test, pred),
    "confusion_matrix [[TN, FP],[FN, TP]]": confusion_matrix(y_test, pred).tolist(),
    "net_value_model": net_value(y_test.values, xgb_test, threshold),
    "net_value_contact_everyone": net_value(y_test.values, np.ones(len(y_test)), 0.5),
    "net_value_contact_nobody": 0.0,
}
print(json.dumps(results["test"], indent=2))
# Don't re-run after tweaking the model: that turns test into validation.

## Step 8: SHAP, why does the model predict churn?

In [ ]:
explainer = shap.TreeExplainer(xgb)
sv = explainer.shap_values(X_test)
plt.figure()
shap.summary_plot(sv, X_test, show=False, max_display=12)
plt.tight_layout(); plt.savefig("shap_summary.png", dpi=120, bbox_inches="tight")

mean_abs = pd.Series(np.abs(sv).mean(axis=0), index=X_test.columns).sort_values(ascending=False)
results["top_shap_features"] = mean_abs.head(8).round(4).to_dict()
print(mean_abs.head(8))

# One customer explained (highest predicted churn risk in the test set)
i = int(np.argmax(xgb_test))
shap.force_plot(explainer.expected_value, sv[i], X_test.iloc[i], matplotlib=True, show=False)
plt.savefig("shap_one_customer.png", dpi=120, bbox_inches="tight")

## Step 9: precision-recall plot + save everything

In [ ]:
p, r, _ = precision_recall_curve(y_test, xgb_test)
plt.figure(figsize=(6, 5)); plt.plot(r, p)
plt.axhline(y_test.mean(), ls="--", c="gray", label="churn rate (random)")
plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("Precision-recall (test)"); plt.legend()
plt.savefig("pr_curve.png", dpi=120, bbox_inches="tight")

json.dump(results, open("results.json", "w"), indent=2)